# TFM DASHBOARD

In [1]:
import pandas as pd

df_eda = pd.read_csv(r"..\00_Data\00_Processed\df_eda.csv")

In [2]:
df_eda['race'] = df_eda['race'].replace({'asian': 'other', 'native american': 'other'})

In [3]:
df_eda.to_csv(r'..\00_Data\00_Processed\df_dashboard.csv', index=False)

In [4]:
df_eval = pd.read_csv(r"..\00_Data\00_Processed\df_eval.csv")

In [5]:
df_eval['sex'] = df_eval['sex'].replace({0: 'male', 1: 'female'})

In [6]:
df_eval_dashboard = df_eval[[
    'person_id','y_true','y_score','y_pred','race', 'sex', 'age_cat'
]]

df_eval_dashboard.to_csv(r'..\00_Data\00_Processed\df_eval_dashboard.csv', index=False)

In [7]:
df_sin_outliers = pd.read_csv(r"..\00_Data\00_Processed\df_sin_outliers.csv")

In [8]:
df_eval = df_eval.merge(
    df_sin_outliers[['person_id','decile_score']],
    on='person_id',
    how='left'
)

df_eval['score_propublica'] = df_eval['decile_score'] / 10

df_eval['y_pred_propublica'] = (df_eval['score_propublica'] >= 0.5).astype(int)

In [9]:
# deciles modelo (robusto)
df_eval['decil_modelo'] = pd.qcut(
    df_eval['y_score'].rank(method='first'),
    10,
    labels=False
)

# deciles propublica
df_eval['decil_propublica'] = pd.qcut(
    df_eval['score_propublica'].rank(method='first'),
    10,
    labels=False
)

df_eval['decil_modelo'] = df_eval['decil_modelo'] + 1
df_eval['decil_propublica'] = df_eval['decil_propublica'] + 1

In [10]:
df_eval_long = pd.concat([

    # 🔵 TU MODELO
    df_eval[['person_id','race','sex','age_cat','y_true','decil_modelo','y_score']]
        .rename(columns={
            'decil_modelo':'decil',
            'y_score':'score'
        })
        .assign(modelo='propuesta_modelo'),

    # 🟠 PROPUBLICA
    df_eval[['person_id','race','sex','age_cat','y_true','decil_propublica','score_propublica']]
        .rename(columns={
            'decil_propublica':'decil',
            'score_propublica':'score'
        })
        .assign(modelo='propublica')

])

In [11]:
df_eval_long['score_bin'] = pd.cut(
    df_eval_long['score'],
    bins=[0,0.2,0.4,0.6,0.8,1]
)

In [12]:
df_eval_long.head()

,person_id,race,sex,age_cat,y_true,decil,score,modelo,score_bin
0,59474.0,other,male,25-45,0,5,0.435356,propuesta_modelo,"(0.4, 0.6]"
1,59890.0,african-american,male,less than 25,1,6,0.489725,propuesta_modelo,"(0.4, 0.6]"
2,61156.0,hispanic,male,less than 25,0,7,0.534697,propuesta_modelo,"(0.4, 0.6]"
3,57039.0,caucasian,female,25-45,0,3,0.360833,propuesta_modelo,"(0.2, 0.4]"
4,58070.0,other,male,less than 25,1,8,0.596857,propuesta_modelo,"(0.4, 0.6]"


In [13]:
# df_lift = pd.concat([
#     df_eval.groupby(['decil_modelo', 'race', 'sex', 'age_cat'])['y_true'].mean().reset_index()
#         .rename(columns={'decil_modelo':'decil','y_true':'recid_rate'})
#         .assign(modelo='propuesta_modelo'),

#     df_eval.groupby(['decil_propublica', 'race', 'sex', 'age_cat'])['y_true'].mean().reset_index()
#         .rename(columns={'decil_propublica':'decil','y_true':'recid_rate'})
#         .assign(modelo='propublica')
# ])

In [14]:
df_fairness_modelo = df_eval.groupby(['race','sex','age_cat']).apply(
    lambda g: pd.Series({
        'FPR': ((g['y_pred']==1)&(g['y_true']==0)).sum() / max((g['y_true']==0).sum(),1),
        'TPR': ((g['y_pred']==1)&(g['y_true']==1)).sum() / max((g['y_true']==1).sum(),1)
    })
).reset_index()

df_fairness_modelo['modelo'] = 'propuesta_modelo'

df_fairness_propublica = df_eval.groupby(['race','sex','age_cat']).apply(
    lambda g: pd.Series({
        'FPR': ((g['y_pred_propublica']==1)&(g['y_true']==0)).sum() / max((g['y_true']==0).sum(),1),
        'TPR': ((g['y_pred_propublica']==1)&(g['y_true']==1)).sum() / max((g['y_true']==1).sum(),1)
    })
).reset_index()

df_fairness_propublica['modelo'] = 'propublica'

df_fairness = pd.concat([
    df_fairness_modelo,
    df_fairness_propublica
])

C:\Users\JAIME\AppData\Local\Temp\ipykernel_138020\1828374102.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_fairness_modelo = df_eval.groupby(['race','sex','age_cat']).apply(
C:\Users\JAIME\AppData\Local\Temp\ipykernel_138020\1828374102.py:10: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_fairness_propublica = df_eval.groupby(['race','sex','age_cat']).apply(


In [15]:
df_fairness['grupo'] = (
    df_fairness['race'] + ' | ' +
    df_fairness['sex'] + ' | ' +
    df_fairness['age_cat']
)

In [16]:
df_fairness.head()

,race,sex,age_cat,FPR,TPR,modelo,grupo
0,african-american,female,25-45,0.272727,0.600000,propuesta_modelo,african-american | female | 25-45
1,african-american,female,46-65,0.000000,0.000000,propuesta_modelo,african-american | female | 46-65
2,african-american,female,less than 25,0.357143,0.600000,propuesta_modelo,african-american | female | less than 25
3,african-american,male,25-45,0.259843,0.666667,propuesta_modelo,african-american | male | 25-45
4,african-american,male,46-65,0.189189,0.423077,propuesta_modelo,african-american | male | 46-65


In [17]:
df_eval_long.to_csv(r'..\00_Data\00_Processed\df_eval_long.csv', index=False)
#df_lift.to_csv(r'..\00_Data\00_Processed\df_lift.csv', index=False)
df_fairness.to_csv(r'..\00_Data\00_Processed\df_fairness.csv', index=False)